<a href="https://colab.research.google.com/github/guitorte/audio/blob/claude/stem-midi-converter-OwEoi/stem-to-midi/notebooks/Stem_to_MIDI_MVP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stem → MIDI — MVP (single track)

Converte **um único stem isolado** (saída do Demucs) em um arquivo MIDI usando [Spotify Basic Pitch](https://github.com/spotify/basic-pitch).

Este MVP valida o pipeline `stem → MIDI` ponta a ponta. A próxima fase será rotear cada tipo de stem para o transcritor especializado (drums → ADTOF, piano → Bytedance, etc — ver README do projeto).

## Convenção de paths no Drive

| | Path |
|---|---|
| Entrada | `/content/drive/MyDrive/stem-to-midi/input/stem.wav` |
| Saída   | `/content/drive/MyDrive/stem-to-midi/output/stem.mid` |

Stems suportados: `vocals`, `bass`, `guitar`, `piano`, `other` (drums **não** — fica para a fase 2).


## 1. Instalar dependências

⚠️ Após rodar esta célula, **reinicie o runtime** (`Runtime ▸ Restart Session`) e siga das células seguintes.

In [ ]:
!pip install -q basic-pitch pretty_midi librosa soundfile mido matplotlib

print('=' * 55)
print('Instalação concluída.')
print('⚠️  Reinicie o runtime e siga adiante.')
print('=' * 55)

## 2. Imports

In [ ]:
import os, warnings
import numpy as np
import matplotlib.pyplot as plt
import librosa, librosa.display
import pretty_midi
from IPython.display import Audio, display, FileLink

warnings.filterwarnings('ignore')

try:
    from basic_pitch.inference import predict as bp_predict
except ImportError as e:
    raise SystemExit('basic_pitch não importou. Rode a célula 1 e reinicie o runtime.') from e

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Colab: {IN_COLAB}')

## 3. Montar Google Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Drive montado.')
else:
    print('Fora do Colab — pulando mount.')

## 4. Configurar paths e tipo de stem

Coloque o arquivo do stem em `/content/drive/MyDrive/stem-to-midi/input/stem.wav` (ou .mp3, .flac, .m4a) e selecione o tipo abaixo. O tipo carrega um preset de parâmetros do Basic Pitch ajustado para aquele instrumento (faixa de frequência, comprimento mínimo de nota, etc).

In [ ]:
# @title Parâmetros
BASE_DIR    = '/content/drive/MyDrive/stem-to-midi'  # @param {type:'string'}
INPUT_NAME  = 'stem.wav'                              # @param {type:'string'}
STEM_TYPE   = 'bass'  # @param ['vocals', 'bass', 'guitar', 'piano', 'other']

INPUT_DIR  = os.path.join(BASE_DIR, 'input')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Aceita extensões alternativas se INPUT_NAME não existir exato
AUDIO_PATH = os.path.join(INPUT_DIR, INPUT_NAME)
if not os.path.exists(AUDIO_PATH):
    stem_base = os.path.splitext(INPUT_NAME)[0]
    for ext in ('.wav', '.mp3', '.flac', '.m4a'):
        candidate = os.path.join(INPUT_DIR, stem_base + ext)
        if os.path.exists(candidate):
            AUDIO_PATH = candidate
            print(f'Arquivo encontrado com extensão alternativa: {os.path.basename(AUDIO_PATH)}')
            break

MIDI_PATH = os.path.join(OUTPUT_DIR, os.path.splitext(os.path.basename(AUDIO_PATH))[0] + '.mid')

print(f'Input audio : {AUDIO_PATH}  (existe: {os.path.exists(AUDIO_PATH)})')
print(f'Output MIDI : {MIDI_PATH}')
print(f'Stem type   : {STEM_TYPE}')

if not os.path.exists(AUDIO_PATH):
    print(f'\n⚠️  Coloque um arquivo em {INPUT_DIR}/ antes de seguir.')
else:
    size_mb = os.path.getsize(AUDIO_PATH) / 1024**2
    print(f'\nTamanho: {size_mb:.2f} MB')
    display(Audio(AUDIO_PATH))

## 5. Carregar presets por tipo de stem

In [ ]:
STEM_PRESETS = {
    'bass':   {'onset_threshold': 0.5, 'frame_threshold': 0.3, 'minimum_note_length':  80.0, 'minimum_frequency':  30.0, 'maximum_frequency':  350.0},
    'vocals': {'onset_threshold': 0.6, 'frame_threshold': 0.3, 'minimum_note_length': 100.0, 'minimum_frequency':  80.0, 'maximum_frequency': 1100.0},
    'guitar': {'onset_threshold': 0.5, 'frame_threshold': 0.3, 'minimum_note_length':  58.0, 'minimum_frequency':  70.0, 'maximum_frequency': 1500.0},
    'piano':  {'onset_threshold': 0.5, 'frame_threshold': 0.3, 'minimum_note_length':  58.0, 'minimum_frequency':  27.5, 'maximum_frequency': 4200.0},
    'other':  {'onset_threshold': 0.5, 'frame_threshold': 0.3, 'minimum_note_length':  58.0},
}

params = STEM_PRESETS[STEM_TYPE]
print(f'Preset "{STEM_TYPE}":')
for k, v in params.items():
    print(f'  {k:22s} = {v}')

## 6. Transcrever stem → MIDI

In [ ]:
assert os.path.exists(AUDIO_PATH), f'Coloque o stem em {AUDIO_PATH} primeiro.'

print(f'Transcrevendo {os.path.basename(AUDIO_PATH)} ...')
_model_output, midi_data, note_events = bp_predict(AUDIO_PATH, **params)
midi_data.write(MIDI_PATH)

n_notes  = sum(len(i.notes) for i in midi_data.instruments)
midi_dur = midi_data.get_end_time()

print(f'\nNotas detectadas : {n_notes}')
print(f'Duração do MIDI  : {midi_dur:.1f} s')
print(f'MIDI salvo em    : {MIDI_PATH}')

if n_notes > 0:
    all_notes = [n for inst in midi_data.instruments for n in inst.notes]
    pitches   = [n.pitch for n in all_notes]
    durations = [n.end - n.start for n in all_notes]
    velocities = [n.velocity for n in all_notes]
    print(f'\nPitch range      : {min(pitches)} – {max(pitches)} (MIDI)')
    print(f'Duração média    : {np.mean(durations):.3f} s')
    print(f'Velocity range   : {min(velocities)} – {max(velocities)}')

## 7. Piano roll do MIDI gerado

In [ ]:
if n_notes == 0:
    print('Nenhuma nota — pule esta célula.')
else:
    roll = midi_data.get_piano_roll(fs=10)
    pmin, pmax = max(0, min(pitches) - 2), min(127, max(pitches) + 2)
    roll_view = roll[pmin:pmax + 1, :]

    NOTE = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(roll_view, aspect='auto', origin='lower', cmap='Blues',
              interpolation='nearest',
              extent=[0, roll_view.shape[1] / 10, pmin, pmax])
    ax.set_xlabel('Tempo (s)')
    ax.set_ylabel('Pitch MIDI')
    ax.set_title(f'Piano roll — {os.path.basename(MIDI_PATH)} ({STEM_TYPE})', fontweight='bold')
    yt = list(range(pmin, pmax + 1, max(1, (pmax - pmin) // 12)))
    ax.set_yticks(yt)
    ax.set_yticklabels([f'{NOTE[p%12]}{p//12 - 1}' for p in yt])
    plt.tight_layout()
    plt.show()

## 8. Preview sonoro do MIDI (sintetizado)

In [ ]:
if n_notes == 0:
    print('Nenhuma nota — sem preview.')
else:
    sr_preview = 22050
    wave = midi_data.synthesize(fs=sr_preview)
    if np.max(np.abs(wave)) > 0:
        wave = 0.9 * wave / np.max(np.abs(wave))
    print('Áudio original (stem):')
    display(Audio(AUDIO_PATH))
    print('MIDI transcrito (sintetizado com onda senoidal):')
    display(Audio(wave, rate=sr_preview))

## 9. Download do MIDI

In [ ]:
print(f'Arquivo salvo em: {MIDI_PATH}')
display(FileLink(MIDI_PATH))

if IN_COLAB:
    try:
        from google.colab import files
        files.download(MIDI_PATH)
    except Exception as e:
        print(f'(download manual via Drive — botão automático falhou: {e})')